# 01_data_exploration.ipynb

Exploratory Data Analysis notebook for dataset statistics, class distribution, numerical distributions, and correlation figures.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config
from src.utils.reproducibility import set_seed
from src.data.loaders import CICIDS2017Loader, UNSWNB15Loader, DatasetNotFoundError
from src.data.validators import DataValidator
from src.data.dataset_adapters import CICIDS2017Adapter, UNSWNB15Adapter

set_seed(42)
config = load_config("../configs/config.yaml")
print(f"Loaded config for: {config.system.name}")


In [ ]:
# 1. Dataset Loading Strategy
try:
    loader = CICIDS2017Loader("../data/raw/CICIDS2017")
    df_raw = loader.load_merged_dataset(sample_frac=0.1)
    print(f"CIC-IDS2017 Loaded: {df_raw.shape}")
except DatasetNotFoundError as e:
    print(f"[NOTE] Raw dataset not found locally:
{e}
")
    print("Generating synthetic mock schema for pipeline structure verification...")
    df_raw = pd.DataFrame({
        " Destination Port": np.random.choice([80, 443, 22, 8080], 1000),
        " Flow Duration": np.random.exponential(1000, 1000),
        " Total Fwd Packets": np.random.randint(1, 100, 1000),
        " Total Backward Packets": np.random.randint(1, 100, 1000),
        " Flow Bytes/s": np.random.uniform(0, 1e6, 1000),
        " Flow Packets/s": np.random.uniform(0, 1e4, 1000),
        " Label": np.random.choice(["BENIGN", "DDoS", "PortScan", "Bot"], 1000, p=[0.7, 0.15, 0.1, 0.05])
    })


In [ ]:
# 2. Data Validation Audit
validator = DataValidator(df_raw, dataset_name="CICIDS2017_EDA", target_col=" Label" if " Label" in df_raw.columns else "Label")
report = validator.run_full_validation()
print("Data Quality Report Summary:")
print(f"- Rows: {report['n_rows']}, Columns: {report['n_columns']}")
print(f"- Missing Cells: {report['missing_values']['total_missing_cells']}")
print(f"- Infinite Cells: {report['infinite_values']['total_infinite_cells']}")
print(f"- Class counts: {report['label_distribution'].get('counts', {})}")


In [ ]:
# 3. Publication-Quality Figures: Class Distribution
fig, ax = plt.subplots(figsize=(10, 5))
adapter = CICIDS2017Adapter(target_column="Label" if "Label" in df_raw.columns else " Label")
X, y_bin, y_multi = adapter.extract_labels(df_raw)

sns.countplot(y=y_multi, order=y_multi.value_counts().index, palette="viridis", ax=ax)
ax.set_title("Class Distribution - CIC-IDS2017 Sample", fontsize=14, fontweight="bold")
ax.set_xlabel("Count", fontsize=12)
ax.set_ylabel("Attack Class", fontsize=12)
plt.tight_layout()
fig_path = Path("../results/figures/01_class_distribution.png")
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved figure to {fig_path}")


In [ ]:
# 4. Correlation Analysis
fig, ax = plt.subplots(figsize=(8, 6))
num_cols = X.select_dtypes(include=[np.number]).columns[:10]  # top 10 numerical features
corr = X[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax, cbar=True)
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
fig_path = Path("../results/figures/01_correlation_matrix.png")
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved correlation plot to {fig_path}")
